In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,r2_score,silhouette_score
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.cluster import KMeans

Data Preprocessing

1. Handle Missing Values

In [ ]:
scard["Income"] = scard["Income"].fillna(scard["Income"].median())

2. Feature engineering

In [ ]:
# Age
scard["Age"] = 2026-scard["Year_Birth"] # use max year from data instead of 2026

In [ ]:
## always convert date in numerical value
# Customer Joining Date
scard["Dt_Customer"] = pd.to_datetime(scard["Dt_Customer"],dayfirst = True) ## Change format of date according to pandas's format

reference_date = scard["Dt_Customer"].max()

scard["Customer_Tenure_Days"] =  (reference_date - scard["Dt_Customer"]).dt.days  ## accessor is a pandas feature that allows you to access datetime properties

In [ ]:
scard.columns
# Spending

scard["Total_Spending"] = scard["MntWines"]+scard["MntFruits"]+scard["MntMeatProducts"]+scard["MntFishProducts"]+scard["MntSweetProducts"]+scard["MntGoldProds"]

In [ ]:
#Children
scard["Total_Children"] = scard["Kidhome"]+scard["Teenhome"]

In [ ]:
# Education 

scard["Education"].value_counts()
scard["Education"]=scard["Education"].replace({"Graduation":"Graduate",
                                               "Basic":"Undergraduate","2n Cycle":"Undergradute",
                                               "PhD":"Postgraduate","Master":"Postgraduate"})

In [ ]:
# Martial Status
scard["Marital_Status"].value_counts()
scard["Living_With"] = scard["Marital_Status"].replace({"Married":"Partner","Together":"Partner",
                                                        "Single":"Alone","Divorced":"Alone",
                                                        "Widow":"Alone","Absurd":"Alone","YOLO":"Alone"})

3. Drop Cplomns

In [ ]:
cols = ["ID","Year_Birth","Marital_Status","Kidhome","Teenhome","Dt_Customer"]
spending_cols = ['MntWines', 'MntFruits','MntMeatProducts', 'MntFishProducts', 'MntSweetProducts','MntGoldProds']

cols_to_del = cols+spending_cols
scard_cleaned = scard.drop(columns = cols_to_del) # store data in another varibles to save your old data as it is

4. Outliers

In [ ]:
col = ["Income","Recency","Response","Age","Total_Spending","Total_Children"]

# relative plots of some features - pair plots
sns.pairplot(scard_cleaned[col]) # for seeing outliers

In [ ]:
# Remove outliers

print("data size with outliers:",len(scard_cleaned))

scard_cleaned = scard_cleaned[(scard_cleaned["Age"] <90)]
scard_cleaned = scard_cleaned[(scard_cleaned["Income"] <600000)]

print("data size without outliers:",len(scard_cleaned))

Heatmap

In [ ]:
corr = scard_cleaned.corr(numeric_only = True)
plt.figure(figsize = (8,6))
sns.heatmap(corr,annot = True,cmap = "coolwarm",annot_kws = {"size":6})

Encoding

In [ ]:
ohe = OneHotEncoder()  # do not pass drop_first in clustring it passes in only linear,logistic

cat_cols = ["Education","Living_With"]
enc_cols = ohe.fit_transform(scard_cleaned[cat_cols])

In [ ]:
en_df = pd.DataFrame(enc_cols.toarray(),columns = ohe.get_feature_names_out(cat_cols),index = scard_cleaned.index)

In [ ]:
df_encoded = pd.concat([scard_cleaned.drop(columns = cat_cols),en_df],axis = 1)

Scaling

In [ ]:
x = df_encoded

sc = StandardScaler()
x_scaled = sc.fit_transform(x)

Visualize

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components = 3)

x_pca = pca.fit_transform(x_scaled)

In [ ]:
# Plot 
fig = plt.figure(figsize = (8,6))
ax = fig.add_subplot(111,projection = "3d")

ax.scatter(x_pca[:,0],x_pca[:,1],x_pca[:,2])

ax.set_xlabel("PCA1")
ax.set_ylabel("PCA2")
ax.set_zlabel("PCA3")
ax.set_title("3d projection")

In [ ]:
pca.explained_variance_ratio_

Analyze K value

1. ELbow Method

In [ ]:
from kneed import KneeLocator

wcss = []
for k in range(1,11):
    kmeans = KMeans(n_clusters = k,random_state = 42)
    kmeans.fit_predict(x_pca)
    wcss.append(kmeans.inertia_)

In [ ]:
knee = KneeLocator(range(1,11),wcss,curve = "convex",direction = "decreasing")
optimal_k = knee.elbow

print("best k value:",optimal_k)
plt.plot(range(1,11),wcss,marker = "o")
plt.xlabel("K")
plt.ylabel("WCSS")

2. SIlhoutte Score

In [ ]:
scores = []

for k in range(2,11):
    kmeans = KMeans(n_clusters = k,random_state = 42)
    labels = kmeans.fit_predict(x_pca)
    score = silhouette_score(x_pca,labels)
    scores.append(score)

#plot
plt.plot(range(2,11),scores,marker = "o")
plt.xlabel("K")
plt.ylabel("silhoutte")

# combined plot
k_range = range(2,11)
fig,ax1 = plt.subplots(figsize = (8,6))

ax1.plot(k_range,wcss[:len(k_range)],marker = "o",color = "blue")
ax1.set_xlabel("K")
ax1.set_ylabel("WCSS")

ax2 = ax1.twinx()
ax2.plot(k_range,scores[:len(k_range)],marker = "x",color = "red",linestyle = "--")
ax2.set_ylabel("SS")

Clustring

In [ ]:
km = KMeans(n_clusters = 4,random_state = 42)
labels_kmeans = km.fit_predict(x_pca)
fig = plt.figure(figsize = (8,6))
ax = fig.add_subplot(111, projection = "3d")

ax.scatter(x_pca[:,0],x_pca[:,1],x_pca[:,2],c = labels_kmeans)
# ax.set_xlabel("PC1")
# ax.set_ylabel("PC2")
# ax.set_zlabel("PC3")
# ax.set_title("3d")

In [ ]:
#Agglomerative
from sklearn.cluster import AgglomerativeClustering

Agg = AgglomerativeClustering(n_clusters = 4,linkage = "ward")
Agg_labels = Agg.fit_predict(x_pca)

fig = plt.figure(figsize = (8,6))
ax = fig.add_subplot(111, projection = "3d")
ax.scatter(x_pca[:,0],x_pca[:,1],x_pca[:,2],c = Agg_labels)

Characterization of Clusters

In [ ]:
x["clusters"] = Agg_labels
pal = ["red","blue","yellow","green"]

sns.countplot(x = x["clusters"],palette = pal,hue = x["clusters"])

In [ ]:
# Income and Spending patterns

sns.scatterplot(x = x["Total_Spending"],y = x["Income"],hue = x["clusters"],palette = pal)

In [ ]:
cluster_summary = x.groupby("clusters").mean()
print(cluster_summary)